In [38]:
from dotenv import load_dotenv
load_dotenv()
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [39]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
def file_reader(directory_path):
    """Reads all PDF files in a directory."""
    
    # ... (Your loading logic) ...
    loader = DirectoryLoader(directory_path, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    
    # --- THE FIX: Convert Objects to Dictionaries ---
    clean_data = []
    for doc in documents:
        clean_data.append({
            "page_content": doc.page_content,
        })
    return clean_data
data=file_reader("/Users/oluwaferanmi/Documents/syllabus_folder")
print(data)

[{'page_content': 'Title:  HIST  102  -  African  Diaspora  Studies  (Spring  2026)  Professor:  Dr.  J.  Hope  Franklin  \nLocation:\n \nDouglass\n \nHall,\n \nRoom\n \n114\n \nReading  Schedule:  Students  are  expected  to  complete  readings  before  the  class  date.  \n●  Reading  1:  "The  Souls  of  Black  Folk"  (Chapters  1-3).  Discussion  on  February  02,  \n2026.\n ●  Reading  2:  "Notes  of  a  Native  Son".  Discussion  on  February  18,  2026.  \nMajor  Assignments:  \n●  Mid-Semester  Research  Paper:  A  10-page  analysis  of  reconstruction  era  policies.  \nDue\n \nMarch\n \n25,\n \n2026\n \nin\n \nclass.\n ●  Group  Presentation:  Slides  due  April  20,  2026.'}, {'page_content': 'Title:  CSCI  350  -  Artificial  Intelligence  (Spring  2026)  Professor:  Dr.  A.  Turing  Schedule:  MWF  \n10:00\n \nAM\n \nCourse  Description:  Introduction  to  agents,  search  algorithms,  and  neural  networks.  \nImportant  Dates  &  Grading:  \n●  Midterm  Exam:  Scheduled 

In [40]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,)


In [41]:
from system_prompt import SYSTEM_PROMPT
# Add this to your existing system prompt string
json_formatting_instruction = """
\n
CRITICAL OUTPUT INSTRUCTION:
You MUST return a JSON Object with exactly one key called "courses".
Do NOT return a raw list.

CORRECT FORMAT:
{{
  "courses": [
      {{ "course_name": "...", "events": [...] }}
  ]
}}

INCORRECT FORMAT:
[
  {{ "course_name": "...", "events": [...] }}
]
"""

# Append it to your main prompt
full_system_prompt = SYSTEM_PROMPT + json_formatting_instruction

In [42]:
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
class Strategy(BaseModel):
    action: str = Field(description="Action to take, e.g. 'Start Studying'")
    start_date: str = Field(description="ISO date YYYY-MM-DD")
    reasoning: str = Field(description="Why this date?")

class Event(BaseModel):
    title: str
    type: str
    due_date: str
    priority: str
    strategy: Optional[Strategy] = None

class SyllabusData(BaseModel):
    course_name: str
    events: List[Event]
# 3. The Single Course (Renamed for clarity)
class CourseSyllabus(BaseModel):
    course_name: str
    events: List[Event]

# 4. THE NEW MASTER CONTAINER
class SemesterPlan(BaseModel):
    courses: List[CourseSyllabus]

In [43]:
from langchain_core.prompts import ChatPromptTemplate
def content_structurer(file_content: List[Dict]):
    prompt = ChatPromptTemplate.from_messages([
        ("system", full_system_prompt),
        ("user", "Here is the syllabus text: {raw_text}")
        ])
    structured_llm = llm.with_structured_output(SemesterPlan, method="json_mode")
    chain=  prompt | structured_llm
    result = chain.invoke({"raw_text": file_content})
    return result.model_dump_json()
structured_data = content_structurer(data)

In [44]:
structured_data

'{"courses":[{"course_name":"HIST 102 - African Diaspora Studies (Spring 2026)","events":[{"title":"Reading 1: The Souls of Black Folk","type":"reading","due_date":"2026-02-02","priority":"low","strategy":{"action":"Read Material","start_date":"2026-02-01","reasoning":"Readings require 1 day lead time."}},{"title":"Reading 2: Notes of a Native Son","type":"reading","due_date":"2026-02-18","priority":"low","strategy":{"action":"Read Material","start_date":"2026-02-17","reasoning":"Readings require 1 day lead time."}},{"title":"Mid-Semester Research Paper","type":"project","due_date":"2026-03-25","priority":"high","strategy":{"action":"Start Drafting","start_date":"2026-03-20","reasoning":"Projects require 5 days lead time."}},{"title":"Group Presentation","type":"assignment","due_date":"2026-04-20","priority":"medium","strategy":{"action":"Start Working","start_date":"2026-04-15","reasoning":"Assignments require 5 days lead time."}}]},{"course_name":"CSCI 350 - Artificial Intelligence (